# RAMP Demand Aggregation for Norte Amazonia Bolivia

This notebook aggregates the RAMP simulation outputs into annual electricity demand (GWh/year) per cluster, formatted as EnergyScope input files (`Demands_C*.csv`).

For each cluster, each municipality's `load_curve_energy_service_full_year_Norte_Amazonia.csv` is read, mapped to EnergyScope end-use categories, and summed across all municipalities in the cluster.

## 1. Setup — clusters and RAMP → EnergyScope mapping

## 0. Control — `Layers_in_out.csv` vs EnergyScope reference

Before building `Demands.csv`, check that the local `Layers_in_out.csv` (used below to convert RAMP electrical energy into useful service energy) matches the reference file used by the EnergyScope model itself, in `EnergyScope_BO_nord_amazonia/Data/2025/sufficiency/00_INDEP/`. A silent divergence here would corrupt every efficiency coefficient computed downstream.

In [1]:
import pandas as pd

LIO_LOCAL_PATH = "../data/Layers_in_out.csv"
LIO_REFERENCE_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/sufficiency/00_INDEP/Layers_in_out.csv"

lio_local = pd.read_csv(LIO_LOCAL_PATH, sep=";", header=0, index_col=0)
lio_reference = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

problems = []

only_local_rows = sorted(set(lio_local.index) - set(lio_reference.index))
only_reference_rows = sorted(set(lio_reference.index) - set(lio_local.index))
if only_local_rows:
    problems.append(f"technologies only in local file: {only_local_rows}")
if only_reference_rows:
    problems.append(f"technologies only in EnergyScope reference: {only_reference_rows}")

only_local_cols = sorted(set(lio_local.columns) - set(lio_reference.columns))
only_reference_cols = sorted(set(lio_reference.columns) - set(lio_local.columns))
if only_local_cols:
    problems.append(f"layers only in local file: {only_local_cols}")
if only_reference_cols:
    problems.append(f"layers only in EnergyScope reference: {only_reference_cols}")

common_rows = sorted(set(lio_local.index) & set(lio_reference.index))
common_cols = sorted(set(lio_local.columns) & set(lio_reference.columns))
changed_rows = sorted(
    tech for tech in common_rows
    if not lio_local.loc[tech, common_cols].equals(lio_reference.loc[tech, common_cols])
)
if changed_rows:
    problems.append(f"technologies with different coefficients: {changed_rows}")

if problems:
    raise ValueError(
        f"Layers_in_out.csv differs from the EnergyScope reference ({LIO_REFERENCE_PATH}): "
        + "; ".join(problems)
    )

print(f"OK — Layers_in_out.csv matches the EnergyScope reference ({LIO_REFERENCE_PATH})")

OK — Layers_in_out.csv matches the EnergyScope reference (../../../EnergyScope_BO_nord_amazonia/Data/2025/sufficiency/00_INDEP/Layers_in_out.csv)


In [2]:
import os
import pandas as pd

OUTPUT_DIR = "output_energyscope"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}
# Maps each RAMP column → (EnergyScope sector, end-use category)
MAPPING = {
    "sufficiency_illumination":            ("HOUSEHOLDS", "LIGHTING_R_C"),
    "sufficiency_ICT":                     ("HOUSEHOLDS", "ELECTRICITY"),
    "sufficiency_cold_storage":            ("HOUSEHOLDS", "FOOD_PRESERVATION"),
    "sufficiency_thermal_comfort":         ("HOUSEHOLDS", "SPACE_COOLING"),
    # Household hot water (electric shower). RAMP outputs FINAL electricity; HEAT_LOW_T_HW in
    # Demands.csv is USEFUL energy. DEC_DIRECT_ELEC in Layers_in_out.csv has ELECTRICITY=-1 and
    # HEAT_LOW_T_DECEN=+1 → 1:1 conversion, so useful = final electricity (coeff=1.0, applied in §2.5).
    "sufficiency_water_heating":           ("HOUSEHOLDS", "HEAT_LOW_T_HW"),
    "big_school_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "big_school_ICT":                      ("SERVICES",   "ELECTRICITY"),
    "big_school_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "big_school_space_cooling":            ("SERVICES",   "SPACE_COOLING"),
    "health_center_illumination":          ("SERVICES",   "LIGHTING_R_C"),
    "health_center_ICT":                   ("SERVICES",   "ELECTRICITY"),
    "health_center_cold_storage":          ("SERVICES",   "FOOD_PRESERVATION"),
    "health_center_space_cooling":         ("SERVICES",   "SPACE_COOLING"),
    "health_center_water_heating":         ("SERVICES",   "HEAT_LOW_T_HW"),
    "health_center_water_supply":          ("SERVICES",   "ELECTRICITY"),
    "health_center_medical_equip":         ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_illumination": ("SERVICES",   "LIGHTING_R_C"),
    "entertainment_business_ICT":          ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_cold_storage": ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "restaurant_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_kitchen":                  ("SERVICES",   "COOKING"),
    "store_illumination":                  ("SERVICES",   "LIGHTING_R_C"),
    "store_ICT":                           ("SERVICES",   "ELECTRICITY"),
    "store_cold_storage":                  ("SERVICES",   "FOOD_PRESERVATION"),
    "workshop_illumination":               ("SERVICES",   "LIGHTING_R_C"),
    "workshop_ICT":                        ("SERVICES",   "ELECTRICITY"),
    "workshop_machinery":                  ("SERVICES",   "MECHANICAL_ENERGY_COMM"),
    "public_lighting_illumination":        ("PUBLIC_LIGHTING", "LIGHTING_P"),
    "rice_processing_rice_processing":     ("INDUSTRY",   "MECHANICAL_ENERGY_IND"),
}

## 2. Helper functions

- **`create_empty_demands()`** — builds the blank EnergyScope demand table (21 rows, all sectors at 0.0), matching the `Demands.csv` format exactly
- **`compute_municipality(muni_name)`** — reads a municipality's RAMP output, maps each column through `MAPPING`, and returns annual GWh per `(sector, end_use)` pair

> Unit conversion: watts × minutes → GWh, i.e. divide by `60 × 10⁹`  
> (RAMP outputs power in W at 1-minute timesteps over 1 year = 525,600 minutes)

In [3]:
def create_empty_demands():
    """Build the empty EnergyScope demand table (21 rows, all sectors at 0.0)."""
    columns = [
        "Category", "Subcategory", "parameter name",
        "HOUSEHOLDS", "SERVICES", "INDUSTRY", "TRANSPORTATION",
        "PUBLIC_LIGHTING", "AGRICULTURE", "MINING", "FISHING_OTHERS",
        "Units"
    ]
    rows = [
        ["Electricity",  "Electricity",                            "ELECTRICITY",                   "[GWh]"],
        ["Lighting",     "Building lighting",                      "LIGHTING_R_C",                  "[GWh]"],
        ["Lighting",     "Public lighting",                        "LIGHTING_P",                    "[GWh]"],
        ["Heat",         "High temperature",                       "HEAT_HIGH_T",                   "[GWh]"],
        ["Heat",         "Space heating",                          "HEAT_LOW_T_SH",                 "[GWh]"],
        ["Heat",         "Hot water",                              "HEAT_LOW_T_HW",                 "[GWh]"],
        ["Heat",         "Cooking",                                "COOKING",                       "[GWh]"],
        ["Cold",         "Process cooling",                        "PROCESS_COOLING",               "[GWh]"],
        ["Cold",         "Space cooling",                          "SPACE_COOLING",                 "[GWh]"],
        ["Cold",         "Food preservation",                      "FOOD_PRESERVATION",             "[GWh]"],
        ["Mobility",     "Passenger",                              "MOBILITY_PASSENGER",            "[Mpkm]"],
        ["Mobility",     "Freight",                                "MOBILITY_FREIGHT",              "[Mtkm]"],
        ["Mobility",     "Long-haul passenger flights",            "AVIATION_LONG_HAUL",            "[Mpkm]"],
        ["Mobility",     "International shipping",                 "SHIPPING",                      "[Mtkm]"],
        ["Mechanical",   "Mechanical energy commercial",           "MECHANICAL_ENERGY_COMM",        "[GWh]"],
        ["Mechanical",   "Mechanical energy industrial",           "MECHANICAL_ENERGY_IND",         "[GWh]"],
        ["Mechanical",   "Mechanical energy agriculture mobility", "MECHANICAL_ENERGY_MOV_AGR",     "[GWh]"],
        ["Mechanical",   "Mechanical energy agriculture fixed",    "MECHANICAL_ENERGY_FIX_AGR",     "[GWh]"],
        ["Mechanical",   "Mechanical energy mining",               "MECHANICAL_ENERGY_MIN",         "[GWh]"],
        ["Mechanical",   "Mechanical energy fishing",              "MECHANICAL_ENERGY_FISH_OTHERS", "[GWh]"],
        ["Non-energy",   "Non-energy",                             "NON_ENERGY",                    "[GWh]"],
    ]
    df = pd.DataFrame(columns=columns)
    for i, row in enumerate(rows):
        df.loc[i] = [row[0], row[1], row[2], 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, row[3]]
    return df


def compute_municipality(muni_name):
    """Read a municipality's RAMP CSV and return {(sector, end_use): annual_GWh}."""
    path = f"data ramp/{muni_name}/load_curve_energy_service_full_year_Norte_Amazonia.csv"
    if not os.path.exists(path):
        # Fail loudly: a missing RAMP file must abort the run. Silently returning {}
        # would amputate a whole municipality from its cluster demand without warning.
        raise FileNotFoundError(
            f"RAMP output missing for '{muni_name}': {path}. "
            f"Cannot aggregate cluster demand with an incomplete municipality set."
        )
    df = pd.read_csv(path)
    results = {}
    for col in df.columns:
        if col in MAPPING:
            key = MAPPING[col]
            annual_gwh = df[col].fillna(0).sum() / 60_000_000_000.0
            results[key] = results.get(key, 0.0) + annual_gwh
    return results

## 2.5. Layer → Demand Coefficient from `Layers_in_out`

RAMP outputs watts of **electricity**; EnergyScope demands are expressed in the **end-use layer** unit (GWh of service).  
For each layer we look up its reference electric technology in `Layers_in_out.csv` and read the `|ELECTRICITY|` input coefficient.

$$\text{demand\_GWh} = \frac{\text{ramp\_electricity\_GWh}}{|\text{coeff}|}$$

Layers mapped to `None` keep `coeff = 1.0` (electricity is already the layer).  
Mobility layers (`MOBILITY_PASSENGER`, `MOBILITY_FREIGHT`, `AVIATION_LONG_HAUL`, `SHIPPING`) are skipped — they are not derived from RAMP electricity.

In [4]:
LAYER_TO_TECH = {
    'ELECTRICITY':                   None,             # Direct, coeff=1.0
    'LIGHTING_R_C':                  'LED_BULB',
    'LIGHTING_P':                    'LED_LIGHT',
    'HEAT_HIGH_T':                   'IND_DIRECT_ELEC',
    'HEAT_LOW_T_SH':                 'DEC_DIRECT_ELEC',
    'HEAT_LOW_T_HW':                 'DEC_DIRECT_ELEC',
    'COOKING':                       'STOVE_ELEC',
    'PROCESS_COOLING':               'IND_ELEC_COLD',
    'SPACE_COOLING':                 'DEC_ELEC_COLD',
    'FOOD_PRESERVATION':             'REFRIGERATOR_EL',
    'MECHANICAL_ENERGY_COMM':        'COMM_MACHINERY_EL',
    'MECHANICAL_ENERGY_IND':         'IND_MACHINERY_EL',
    'MECHANICAL_ENERGY_MOV_AGR':     'TRACTOR_EL',
    'MECHANICAL_ENERGY_FIX_AGR':     'AGR_MACHINERY_EL',
    'MECHANICAL_ENERGY_MIN':         'MIN_MACHINERY_EL',
    'MECHANICAL_ENERGY_FISH_OTHERS': 'FISH_MACHINERY_EL',
    'NON_ENERGY':                    None,
}

MOBILITY_LAYERS = {'MOBILITY_PASSENGER', 'MOBILITY_FREIGHT', 'AVIATION_LONG_HAUL', 'SHIPPING'}    # no need to convert from RAMP electricity, already in layer units

# Load Layers_in_out — index = technology name, columns = layer names
lio = pd.read_csv("../data/Layers_in_out.csv", sep=";", header=0, index_col=0)

# Build LAYER_TO_COEFF: end-use layer → |ELECTRICITY coefficient|
# For None entries coeff=1.0 (RAMP output already in layer units).
# Mobility layers are excluded — they are not derived from RAMP electricity.
LAYER_TO_COEFF = {}
for layer, tech in LAYER_TO_TECH.items():
    if tech is None:
        LAYER_TO_COEFF[layer] = 1.0
    else:
        LAYER_TO_COEFF[layer] = abs(float(lio.loc[tech, "ELECTRICITY"]))

print("Layer → |ELECTRICITY coefficient|:")
for layer, coeff in LAYER_TO_COEFF.items():
    print(f"  {layer:<35} {coeff:.6f}")

Layer → |ELECTRICITY coefficient|:
  ELECTRICITY                         1.000000
  LIGHTING_R_C                        2.941176
  LIGHTING_P                          2.941176
  HEAT_HIGH_T                         1.000000
  HEAT_LOW_T_SH                       1.000000
  HEAT_LOW_T_HW                       1.000000
  COOKING                             1.000000
  PROCESS_COOLING                     0.496500
  SPACE_COOLING                       0.400000
  FOOD_PRESERVATION                   2.792308
  MECHANICAL_ENERGY_COMM              1.212121
  MECHANICAL_ENERGY_IND               1.111111
  MECHANICAL_ENERGY_MOV_AGR           1.111111
  MECHANICAL_ENERGY_FIX_AGR           1.111111
  MECHANICAL_ENERGY_MIN               1.111111
  MECHANICAL_ENERGY_FISH_OTHERS       1.111111
  NON_ENERGY                          1.000000


## 2.6. Cooking Demand — Households (non-RAMP)

Household cooking demand is **not captured by RAMP** because traditional firewood and LPG stoves are non-electric appliances and were excluded from the electrical load simulation. This section quantifies that missing demand using census data and a fuel-independent useful energy intensity.

### Data sources
- **Household counts**: Bolivia National Census 2024 (`CSV_final.csv`) — `cooking_hh = total_2024 − no_cocina_2024` per municipality, i.e. every household that uses any cooking fuel (Leña, Guano/bosta/taquia, Energía solar, Gas domiciliario, Gas en garrafa, Electricidad, Otro). Only "No cocina" is excluded. Region total: **82,501 cooking households**.
- **Useful energy intensity**: 1,344 kWh/household/year (0.001344023 GWh) — fuel-independent expert estimate from Pablo Jimenez Zabalaga, as used in Roger Arias's thesis (2024–2025). Consistent with the wood-derived value from Hallberg & Hallme (2015): 66.7 MJ/day ÷ 6.25 ≈ 1,110 kWh/year useful (same order of magnitude).

### Formula
$$\text{GWh}_{\text{useful}} = n_{\text{cooking\_hh}} \times 0.001344023$$

where $n_{\text{cooking\_hh}} = \text{total\_2024} - \text{no\_cocina\_2024}$ per municipality.

This value represents **useful** cooking energy. EnergyScope's optimizer then determines the fuel mix (STOVE_WOOD, STOVE_LPG, STOVE_ELEC) required to deliver that useful demand, using the efficiency coefficients in `Layers_in_out.csv`.

The result is added to **HOUSEHOLDS → COOKING** in each cluster's `Demands` table.  
**SERVICES → COOKING** (restaurant kitchen demand already in RAMP) is left untouched.

In [5]:
# Bolivia Census 2024 — cooking households per municipality
# cooking_hh = total_2024 − no_cocina_2024 (all fuel types except "No cocina")
# Source: CSV_final.csv, "NÚMERO DE VIVIENDAS SEGÚN ENERGÍA UTILIZADA PARA COCINAR | 2024"
COOKING_DATA = {
    # municipality_name : cooking_hh (total_2024 - no_cocina_2024)
    "Ixiamas":               3225,   # 3306 - 81
    "Riberalta":             26940,  # 27442 - 502
    "Guayaramerín":          10670,  # 10891 - 221
    "Reyes":                 3353,   # 3417 - 64
    "Santa_Rosa_Beni":       2696,   # 2755 - 59
    "Exaltación":            1436,   # 1455 - 19
    "Cobija":                15074,  # 15564 - 490
    "Porvenir":              2200,   # 2230 - 30
    "Bolpebra":              795,    # 802 - 7
    "Bella_Flor":            1223,   # 1235 - 12
    "Puerto_Rico":           1933,   # 1989 - 56
    "San_Pedro":             609,    # 612 - 3
    "Filadelfia":            2479,   # 2514 - 35
    "Puerto_Gonzalo_Moreno": 1948,   # 1965 - 17
    "San_Lorenzo":           1822,   # 1850 - 28
    "Sena":                  2815,   # 2875 - 60
    "Santa_Rosa_Pando":      853,    # 860 - 7
    "Ingavi":                643,    # 649 - 6
    "Nueva_Esperanza":       488,    # 489 - 1
    "Villa_Nueva":           701,    # 708 - 7
    "Santos_Mercado":        598,    # 601 - 3
}
# Total cooking households across region: 82,501 (= 84,209 total − 1,708 no_cocina)

# Fuel-independent useful cooking energy per household per year
# Source: Pablo Jimenez Zabalaga expert estimate (Roger Arias thesis, 2024-2025)
# Consistent with Hallberg & Hallme (2015): 66.7 MJ/day / 6.25 ≈ 1,110 kWh/year useful
USEFUL_COOKING_PER_HH_GWh = 0.001344023  # GWh/household/year


def compute_cooking_demand(municipality_name):
    """Return annual household cooking demand in GWh (useful service) for one municipality.

    Applies a single fuel-independent useful energy to ALL cooking households
    (any fuel except "No cocina"). EnergyScope optimization then allocates the
    fuel mix via STOVE_WOOD/STOVE_LPG/STOVE_ELEC efficiency coefficients.
    """
    if municipality_name not in COOKING_DATA:
        print(f"Warning: '{municipality_name}' not found in COOKING_DATA — cooking demand set to 0.")
        return 0.0

    return COOKING_DATA[municipality_name] * USEFUL_COOKING_PER_HH_GWh

## 3. Aggregate by cluster

Loops over each cluster, sums the RAMP demand across all its municipalities, and saves the result to `output_energyscope/C{k}/Demands_C{k}.csv`.

In [6]:
print("--- STARTING ANALYSIS ---")

for cluster_id, municipalities in CLUSTERS.items():
    df_cluster = create_empty_demands()

    total_gwh = 0.0
    by_sector = {"HOUSEHOLDS": 0.0, "SERVICES": 0.0, "INDUSTRY": 0.0, "PUBLIC_LIGHTING": 0.0}

    for muni in municipalities:
        for (sector, end_use), elec_gwh in compute_municipality(muni).items():
            if end_use in MOBILITY_LAYERS:
                continue
            coeff = LAYER_TO_COEFF.get(end_use, 1.0)
            demand_gwh = elec_gwh / coeff
            row_filter = df_cluster["parameter name"] == end_use
            df_cluster.loc[row_filter, sector] += demand_gwh
            total_gwh += demand_gwh
            if sector in by_sector:
                by_sector[sector] += demand_gwh

        # Add household cooking demand (wood + LPG + electric stoves)
        # Traditional firewood/LPG cooking is non-electric and not captured by RAMP
        cooking_gwh = compute_cooking_demand(muni)
        cooking_row = df_cluster["parameter name"] == "COOKING"
        current_cooking = df_cluster.loc[cooking_row, "HOUSEHOLDS"].fillna(0).values[0]
        df_cluster.loc[cooking_row, "HOUSEHOLDS"] = current_cooking + cooking_gwh
        total_gwh += cooking_gwh
        by_sector["HOUSEHOLDS"] += cooking_gwh

    output_file = f"{OUTPUT_DIR}/C{cluster_id}/Demands.csv"
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df_cluster.to_csv(output_file, sep=";", index=False)

    print(f"Cluster {cluster_id} ({len(municipalities)} municipalities):")
    print(f"  Total Demand : {total_gwh:.4f} GWh/year")
    for sector, val in by_sector.items():
        print(f"  - {sector:<15}: {val:.4f} GWh")
    print("-" * 40)

--- STARTING ANALYSIS ---
Cluster 1 (4 municipalities):
  Total Demand : 31.4472 GWh/year
  - HOUSEHOLDS     : 30.4257 GWh
  - SERVICES       : 0.8157 GWh
  - INDUSTRY       : 0.1779 GWh
  - PUBLIC_LIGHTING: 0.0280 GWh
----------------------------------------
Cluster 2 (1 municipalities):
  Total Demand : 2.3434 GWh/year
  - HOUSEHOLDS     : 2.2645 GWh
  - SERVICES       : 0.0635 GWh
  - INDUSTRY       : 0.0133 GWh
  - PUBLIC_LIGHTING: 0.0021 GWh
----------------------------------------
Cluster 3 (3 municipalities):
  Total Demand : 117.9286 GWh/year
  - HOUSEHOLDS     : 114.0469 GWh
  - SERVICES       : 3.1087 GWh
  - INDUSTRY       : 0.6697 GWh
  - PUBLIC_LIGHTING: 0.1033 GWh
----------------------------------------
Cluster 4 (12 municipalities):
  Total Demand : 48.5021 GWh/year
  - HOUSEHOLDS     : 46.9323 GWh
  - SERVICES       : 1.2630 GWh
  - INDUSTRY       : 0.2644 GWh
  - PUBLIC_LIGHTING: 0.0424 GWh
----------------------------------------
Cluster 5 (1 municipalities):
  Total